In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Sun Jul 12 18:59:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
  "transformers==4.53.3" \
  "peft==0.17.1" \
  "trl" \
  "accelerate" \
  "bitsandbytes" \
  "wandb"

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset, Dataset

PyTorch version: 2.11.0+cu128
Transformers version: 4.53.3
PEFT version: 0.17.1


# Configurations

In [9]:
# Run configuration
SEED = 42
LANG = 'vi'  # e.g., 'en' | 'ja' | 'id'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260711111235'
LORA_CKPT_DIR = 'checkpoint-120'

# Data configuration
TEST_SIZE = 10
DATA_ID = 'google/xquad'
DATA_DIR = 'xquad.{lang}'
DATA_SPLIT = 'validation'

# Utilities

In [6]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size,
    data_id=DATA_ID,
    data_dir=DATA_DIR,
    data_split=DATA_SPLIT,
):
    assert '{lang}' in data_dir, "Data directory must contain a '{lang}' placeholder."
    
    dataset_stream = load_dataset(
        data_id,
        data_dir=data_dir.format(lang=lang),
        split=data_split,
        streaming=True,
    )

    test_data = []

    for i, example in enumerate(dataset_stream):
        if i < size:
            test_data.append(example)
        else:
            break

    return Dataset.from_list(test_data)

# Model

In [7]:
# Load the tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

print()
print(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(



XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [8]:
# Load LoRA model
lora_model = PeftModel.from_pretrained(base_model, 
                                       subfolder=LORA_CKPT_DIR,
                                       model_id=LORA_ID)
lora_model = lora_model.to(DEVICE).eval()

print("device:", lora_model.device)
print()
print(lora_model)

device: cuda:0

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): XLMRobertaForQuestionAnswering(
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250002, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): XLMRobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x XLMRobertaLayer(
              (attention): XLMRobertaAttention(
                (self): XLMRobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
     

# Inference

In [ ]:
dataset = load_test_dataset(LANG, size=TEST_SIZE)
print(dataset)

Dataset({
    features: ['id', 'context', 'question', 'answers'],
    num_rows: 10
})


In [19]:
# Inference
for i, example in enumerate(dataset):
    # Tokenize
    inputs = tokenizer(
        example['question'],
        example['context'],
        truncation='only_second',
        max_length=384,
        return_tensors='pt',
    ).to(DEVICE)

    # Predict
    with torch.no_grad():
        outputs = lora_model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

    # Decode answer span
    start_idx = torch.argmax(start_logits)
    end_idx = torch.argmax(end_logits)
    answer_ids = inputs['input_ids'][0, start_idx : end_idx + 1]
    pred_answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    # Ground truth
    gt_answer = example['answers']['text'][0]

    print(f"---- Example {i+1} ----")
    print(f"Question: {example['question']}")
    print(f"Ground truth: {gt_answer}")
    print(f"Predicted: {pred_answer}")
    print()

---- Example 1 ----
Question: Đội thủ Panthers đã thua bao nhiêu điểm?
Ground truth: 308
Predicted: 308

---- Example 2 ----
Question: Jared Allen có bao nhiêu lần vật ngã trong sự nghiệp?
Ground truth: 136
Predicted: 61⁄2

---- Example 3 ----
Question: Luke Kuechly đã có bao nhiêu cú húc?
Ground truth: 118
Predicted: 11 lần, đồng thời húc văng bóng (fumble) 3 lần và lấy lại được bóng (recover) 2 lần. Đồng nghiệp lineman Mario Addison đã thêm 61⁄2 lần vật ngã. Đội hình Panthers cũng có người tiền vệ (defensive end) kỳ cựu Jared Allen, người tham gia Pro Bowl 5 lần, dẫn đầu về số lần vật ngã trong sự nghiệp NFL với 136 lần, cùng với người tiền vệ Kony Ealy, người đã có 5 lần vật ngã sau 9 lần xuất phát. Phía sau họ, hai trong số ba người hàng vệ (linebacker) xuất phát của Panthers cũng được chọn để chơi trong Pro Bowl: Thomas Davis và Luke Kuechly. Davis đã có 51⁄2 lần vật ngã, 4 lần húc văng bóng và 4 lần đoạt bóng, trong khi Kuechly dẫn đầu đội về số lần húc (118 lần), 2 lần húc văng 